In [ ]:
%%time
import tables
from ctapipe.visualization import CameraDisplay
from ctapipe.coordinates import EngineeringCameraFrame
from ctapipe_io_lst import LSTEventSource
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import RANSACRegressor, LinearRegression


# CAMERA GEOMETRY
sa = LSTEventSource.create_subarray(tel_id=1)
focal_length = sa.tel[1].optics.equivalent_focal_length
camera_geom = sa.tel[1].camera.geometry

In [ ]:
# DL1 DATACHECK FILE
#a = tables.open_file("/data/cta/users-ifae/moralejo/CTA/summer_student_2026/datacheck/ALL_h5/datacheck_dl1_LST-1.Run20714.h5")
a = tables.open_file("/data/cta/users-ifae/moralejo/CTA/summer_student_2026/datacheck/datacheck_dl1_LST-1.Run24704.h5")

# We'll first focus on looking for clues at the 'cosmics' data. 
# flatfields and pedestals will be later checked
# one can take a look at the content at the cheatsheet

In [ ]:
a.root.dl1datacheck.cosmics.col('subrun_index').shape
a.root.dl1datacheck.cosmics.col('elapsed_time').shape

In [ ]:
%%time

# USEFUL VARIABLES
subruns = (a.root.dl1datacheck.cosmics.col('subrun_index')) #0,1,...,57 number of subruns in this run
n_subruns = len(subruns)
time = a.root.dl1datacheck.cosmics.col('elapsed_time')

# ROBUST FIT FUNCTION
# x: 1D array like subruns 58
# y: 2D array like cog_pixel 58 x 1855
# fit : 2D array 58 x 1855

def robust_fit(x, y):
    X = x.astype(float).reshape(-1, 1)
    fit = np.zeros_like(y, dtype=float)
    for p in range(y.shape[1]):
        yp = y[:, p]
        # First fit on all points.
        model = LinearRegression()
        model.fit(X, yp)
        # Keep only points that are not too far from the first fit.
        residual = np.abs(yp - model.predict(X))
        mad = np.median(np.abs(residual - np.median(residual)))
        cutoff = max(4.0 * mad, 3.0)
        good = residual <= cutoff
        # Refit using only the kept points.
        if good.sum() >= 2:
            model = LinearRegression()
            model.fit(X[good], yp[good])
            fit[:, p] = model.predict(X)
        else:
            fit[:, p] = np.median(yp)
    return fit



# FILTER 1: EVENT RATE--------------------------------------------------------------------------------------------
# we expect the event rate to present anomalies when a satellite comes in the FoV
# for a really bright one, the pixels in the camera might de-activate as a safety measure, making the rate drop
# for a sat not that bright, there might be a peak in the rate


num_events = a.root.dl1datacheck.cosmics.col('num_events') # events in each subrun 1D , 58
event_rate = num_events / time  # 58/58
event_rate_sigma = np.sqrt(np.maximum(num_events, 1.0)) / time  

# ROBUST FIT
event_rate_fit = robust_fit(subruns, event_rate.reshape(-1, 1)).flatten()  #in here we are setting y.shape = (58,1) which sets the loop in robust fit to 1 iteration

# Selection criteria
erate_z_score = (event_rate - event_rate_fit) / event_rate_sigma
checks_1 = subruns[ np.abs(erate_z_score) > 3]

print('Subruns to check =', checks_1)


# FILTER 2: EVENT RATE VS INTENSITY---------------------------------------------------------------------------------
# we expect the event rate per intensity distribuition to vary for an anomali with respect to the NSB

# Intensity = sum of charge over all the pixels
# Hist_intensity: for each subrun, contains the histogram of intensities
hist_intensity = a.root.dl1datacheck.cosmics.col('hist_intensity')

# Find the center of each bin for the histogram of intensity
intensity_bins = a.root.dl1datacheck.histogram_binning.col('hist_intensity')[0]
intensity = 0.5*(intensity_bins[1:]+intensity_bins[:-1])  

# Event rate per intensity value in each subrun---------------------------------------------------------------------
# Therefore, to perform a more sensitive analysis we'll implement the Kolmogorov-Smirnov statistic, because we want to compare the whole shape of our distribuition
# with that end we implement the Kolmogorov-Smirnov parameter D = max |F(x) - Fn| which tells us how different are two cumulative distribuition functions
# we will compare the CDF for each subrun with its neighbours, bc the satellite is expected to crossthe FoV within a fine period of time

cdf = np.cumsum(hist_intensity / hist_intensity.sum(axis=1, keepdims=True), axis=1) #we need to normalize the histogram. 1 cdf per subrun
D = np.zeros(n_subruns) 

for s in range(0, n_subruns-1):
    D[s] = np.max(np.abs(cdf[s] - cdf[s+1])) #we compare the CDF in each subrun 's' with its neighbouring one 's+1'

# This is not a Poisson process therefore we cannot treat it as such


sigma_2 = np.std(D)
deviations_2 = np.abs(D - np.median(D))/sigma_2

checks_2 = subruns[deviations_2 > 1]
print('Subruns to check =', checks_2)